# ETL dos dados do KOPPEN - 2013

**Fonte**: https://forest-gis.com/classificacao-climatica-de-koppen-geiger-em-shapefile/

## Setup

In [127]:
# Necessário pandas `pip install pandas`
# -- Imports --
import pandas as pd
import janitor

In [ ]:
# -- Variáveis --
df_controle = pd.read_csv("../municipios_brasil_original.csv",decimal=",", index_col=1)
df = pd.read_excel("../koppen/Koppen Brazilian municipalities.xls", sheet_name="Data", index_col = 1)
# Rio de Janeiro é duplicado...
df.drop_duplicates(inplace=True)

df_join = df_controle.join(df)
df_join['koppen_cod'] = df_join['Köppen'].astype('category').cat.codes

df_join.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 1600105 to 3205309
Data columns (total 35 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   municipio_nome     40 non-null     object 
 1   temperatura_anual  20 non-null     float64
 2   populacao          40 non-null     float64
 3   IDHM               40 non-null     int64  
 4   PIB_per_capita     40 non-null     float64
 5   Municipality       38 non-null     object 
 6   State              38 non-null     object 
 7   Region             38 non-null     object 
 8   Köppen             38 non-null     object 
 9   Altitude           38 non-null     float64
 10  T_jan              38 non-null     float64
 11  T_feb              38 non-null     float64
 12  T_mar              38 non-null     float64
 13  T_apr              38 non-null     float64
 14  T_may              38 non-null     float64
 15  T_jun              38 non-null     float64
 16  T_jul              38 

In [114]:
df_join.head()

,municipio_nome,temperatura_anual,populacao,IDHM,PIB_per_capita,Municipality,State,Region,Köppen,Altitude,...,R_apr,R_may,R_jun,R_jul,R_aug,R_sep,R_oct,R_nov,R_dec,koppen_cod
municipio_id,,,,,,,,,,,,,,,,,,,,,
1600105,Amapa,26.0,733.8,642,27.6,Amapá,AP,Norte,Am,32.977505,...,454.994385,502.083496,352.574951,266.872375,131.634979,50.952412,36.023254,74.646660,179.130753,1
2800308,Aracaju,NaN,602.8,770,37.0,Aracaju,SE,Nordeste,Am,7.911602,...,210.083328,283.561096,215.044449,200.616669,120.288887,90.105553,58.972221,56.205555,46.933334,1
1702208,Araguatins,24.0,31.9,631,14.5,Araguatins,TO,Norte,Aw,152.335342,...,238.345474,81.163727,24.790838,7.476906,11.412317,47.509575,86.993240,143.903122,212.113403,3
3506003,Bauru,22.0,379.0,801,54.5,Bauru,SP,Sudeste,Cfa,532.107910,...,64.961273,56.240665,53.965424,40.426003,32.031811,68.899033,129.287689,153.582291,226.764862,4
1501402,Belém,NaN,1303.4,746,31.1,Belém,PA,Norte,Am,8.206128,...,443.886719,354.629517,179.579391,170.070572,129.484680,97.110489,62.026928,93.887650,185.294342,1


## Temperatura anual das cidades

In [128]:
df_final: pd.DataFrame = df_join[['municipio_nome','PIB_per_capita','populacao', 'IDHM', 'Altitude', 'temperatura_anual', 'koppen_cod']]
df_final

,municipio_nome,PIB_per_capita,populacao,IDHM,Altitude,temperatura_anual,koppen_cod
municipio_id,,,,,,,
1600105,Amapa,27.6,733.8,642,32.977505,26.0,1
2800308,Aracaju,37.0,602.8,770,7.911602,NaN,1
1702208,Araguatins,14.5,31.9,631,152.335342,24.0,3
3506003,Bauru,54.5,379.0,801,532.107910,22.0,4
1501402,Belém,31.1,1303.4,746,8.206128,NaN,1
3106200,Belo Horizonte,56.2,2315.6,810,909.062683,23.0,6
1400100,Boa Vista,44.1,413.5,752,87.631096,NaN,1
5300108,Brasília,129.8,2817.3,824,1034.792358,21.0,3
5002704,Campo Grande,47.1,898.1,784,467.670349,NaN,1


In [129]:
nova_temp = df_join.filter(like='T_').mean(axis=1)
df_final['temperatura_anual'] = nova_temp.combine_first(df_join['temperatura_anual'])

/tmp/ipykernel_89520/3081447080.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['temperatura_anual'] = nova_temp.combine_first(df_join['temperatura_anual'])


In [130]:
df_final

,municipio_nome,PIB_per_capita,populacao,IDHM,Altitude,temperatura_anual,koppen_cod
municipio_id,,,,,,,
1600105,Amapa,27.6,733.8,642,32.977505,27.284206,1
2800308,Aracaju,37.0,602.8,770,7.911602,25.950050,1
1702208,Araguatins,14.5,31.9,631,152.335342,27.405858,3
3506003,Bauru,54.5,379.0,801,532.107910,20.934916,4
1501402,Belém,31.1,1303.4,746,8.206128,27.934292,1
3106200,Belo Horizonte,56.2,2315.6,810,909.062683,19.095724,6
1400100,Boa Vista,44.1,413.5,752,87.631096,25.424889,1
5300108,Brasília,129.8,2817.3,824,1034.792358,20.960918,3
5002704,Campo Grande,47.1,898.1,784,467.670349,22.322037,1


In [133]:
df_final['chuva_anual'] = df_join.filter(like='R_').mean(axis=1)

/tmp/ipykernel_89520/807827021.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['chuva_anual'] = df_join.filter(like='R_').mean(axis=1)


## Conserta nome e exporta CSV

In [136]:
df_final.columns = df_final.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

In [137]:
df_final.dropna(inplace=True)
df_final

/tmp/ipykernel_89520/3494360828.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final.dropna(inplace=True)


,municipio_nome,pib_per_capita,populacao,idhm,altitude,temperatura_anual,koppen_cod,chuva_anual
municipio_id,,,,,,,,
1600105,Amapa,27.6,733.8,642,32.977505,27.284206,1,265.984365
2800308,Aracaju,37.0,602.8,770,7.911602,25.950050,1,125.298610
1702208,Araguatins,14.5,31.9,631,152.335342,27.405858,3,141.153402
3506003,Bauru,54.5,379.0,801,532.107910,20.934916,4,115.753342
1501402,Belém,31.1,1303.4,746,8.206128,27.934292,1,250.650572
3106200,Belo Horizonte,56.2,2315.6,810,909.062683,19.095724,6,128.863013
1400100,Boa Vista,44.1,413.5,752,87.631096,25.424889,1,150.697335
5300108,Brasília,129.8,2817.3,824,1034.792358,20.960918,3,127.622327
5002704,Campo Grande,47.1,898.1,784,467.670349,22.322037,1,134.005501


In [139]:
df_final.to_csv('../municipios_brasil.csv', decimal=',')